# Lab 2: Apply Overfitting Solutions

**Topic 2: Solving Overfitting Issues with Vibe Coding**

In this lab, we apply various regularization techniques to combat overfitting, test each technique individually, and combine them into one regularized model.

## Techniques Covered
- Dropout
- Batch Normalization
- Data Augmentation (RandomFlip, RandomRotation)
- Early Stopping
- L2 Regularization

In [ ]:
# Run this cell in Google Colab to install dependencies
# Skip if running locally with uv
import sys
if 'google.colab' in sys.modules:
    !pip install -q keras torch torchvision python-dotenv datasets transformers huggingface_hub
    print('Dependencies installed!')

## 1. Environment Setup

## 2. Load and Preprocess Data

In [ ]:
# Load Fashion-MNIST
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

# Normalize to [0, 1]
x_train_full = x_train_full.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Flat version for dense models
x_train_flat = x_train_full[:50000].reshape(-1, 784)
x_val_flat = x_train_full[50000:].reshape(-1, 784)

# Image version for CNN/augmentation models (add channel dimension)
x_train_img = x_train_full[:50000].reshape(-1, 28, 28, 1)
x_val_img = x_train_full[50000:].reshape(-1, 28, 28, 1)
x_test_img = x_test.reshape(-1, 28, 28, 1)

y_train = y_train_full[:50000]
y_val = y_train_full[50000:]

print(f"Training set (flat): {x_train_flat.shape}")
print(f"Training set (image): {x_train_img.shape}")
print(f"Validation set: {x_val_flat.shape}")

## 3. Baseline Model (No Regularization)

Same overfitting-prone model from Lab 1 as our baseline for comparison.

In [ ]:
EPOCHS = 30
BATCH_SIZE = 128

def build_baseline_model():
    """Baseline: large dense network, no regularization."""
    model = keras.Sequential([
        keras.layers.Input(shape=(784,)),
        keras.layers.Dense(512, activation="relu"),
        keras.layers.Dense(512, activation="relu"),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dense(10, activation="softmax"),
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

print("Training baseline model...")
baseline_model = build_baseline_model()
baseline_history = baseline_model.fit(
    x_train_flat, y_train,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_data=(x_val_flat, y_val),
    verbose=1,
)

## 4. Technique 1: Dropout

Dropout randomly sets a fraction of input units to 0 during training, preventing co-adaptation of neurons.

In [ ]:
def build_dropout_model():
    """Model with Dropout(0.5) after each hidden layer."""
    model = keras.Sequential([
        keras.layers.Input(shape=(784,)),
        keras.layers.Dense(512, activation="relu"),
        keras.layers.Dropout(0.5),
        keras.layers.Dense(512, activation="relu"),
        keras.layers.Dropout(0.5),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dropout(0.4),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dropout(0.4),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(10, activation="softmax"),
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

print("Training model with Dropout...")
dropout_model = build_dropout_model()
dropout_history = dropout_model.fit(
    x_train_flat, y_train,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_data=(x_val_flat, y_val),
    verbose=1,
)

## 5. Technique 2: Batch Normalization

Batch Normalization normalizes layer inputs, stabilizing and accelerating training. It also provides a mild regularization effect.

In [ ]:
def build_batchnorm_model():
    """Model with BatchNormalization before each activation."""
    model = keras.Sequential([
        keras.layers.Input(shape=(784,)),
        keras.layers.Dense(512),
        keras.layers.BatchNormalization(),
        keras.layers.Activation("relu"),
        keras.layers.Dense(512),
        keras.layers.BatchNormalization(),
        keras.layers.Activation("relu"),
        keras.layers.Dense(256),
        keras.layers.BatchNormalization(),
        keras.layers.Activation("relu"),
        keras.layers.Dense(256),
        keras.layers.BatchNormalization(),
        keras.layers.Activation("relu"),
        keras.layers.Dense(128),
        keras.layers.BatchNormalization(),
        keras.layers.Activation("relu"),
        keras.layers.Dense(10, activation="softmax"),
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

print("Training model with BatchNormalization...")
batchnorm_model = build_batchnorm_model()
batchnorm_history = batchnorm_model.fit(
    x_train_flat, y_train,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_data=(x_val_flat, y_val),
    verbose=1,
)

## 6. Technique 3: Data Augmentation

Data augmentation artificially increases the effective size of the training set by applying random transformations. We use a CNN architecture here to support spatial augmentations.

In [ ]:
def build_augmentation_model():
    """CNN with data augmentation layers."""
    model = keras.Sequential([
        keras.layers.Input(shape=(28, 28, 1)),
        # Data augmentation layers (only active during training)
        keras.layers.RandomFlip("horizontal"),
        keras.layers.RandomRotation(0.1),
        # CNN feature extraction
        keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Flatten(),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dense(10, activation="softmax"),
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

print("Training model with Data Augmentation...")
augmentation_model = build_augmentation_model()
augmentation_history = augmentation_model.fit(
    x_train_img, y_train,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_data=(x_val_img, y_val),
    verbose=1,
)

## 7. Technique 4: Early Stopping

Early Stopping monitors validation loss and stops training when it stops improving, restoring the best weights.

In [ ]:
print("Training baseline model with Early Stopping...")
early_stop_model = build_baseline_model()

early_stopping_cb = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1,
)

early_stop_history = early_stop_model.fit(
    x_train_flat, y_train,
    epochs=60,  # Set high, early stopping will halt it
    batch_size=BATCH_SIZE,
    validation_data=(x_val_flat, y_val),
    callbacks=[early_stopping_cb],
    verbose=1,
)

print(f"\nEarly stopping halted training at epoch {len(early_stop_history.history['loss'])}")

## 8. Technique 5: L2 Regularization

L2 regularization adds a penalty proportional to the square of the weights, discouraging large weight values.

In [ ]:
def build_l2_model():
    """Model with L2 weight regularization on all dense layers."""
    l2_reg = keras.regularizers.l2(1e-4)
    model = keras.Sequential([
        keras.layers.Input(shape=(784,)),
        keras.layers.Dense(512, activation="relu", kernel_regularizer=l2_reg),
        keras.layers.Dense(512, activation="relu", kernel_regularizer=l2_reg),
        keras.layers.Dense(256, activation="relu", kernel_regularizer=l2_reg),
        keras.layers.Dense(256, activation="relu", kernel_regularizer=l2_reg),
        keras.layers.Dense(128, activation="relu", kernel_regularizer=l2_reg),
        keras.layers.Dense(10, activation="softmax"),
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

print("Training model with L2 Regularization...")
l2_model = build_l2_model()
l2_history = l2_model.fit(
    x_train_flat, y_train,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_data=(x_val_flat, y_val),
    verbose=1,
)

## 9. Compare All Individual Techniques

In [ ]:
histories = {
    "Baseline": baseline_history.history,
    "Dropout": dropout_history.history,
    "BatchNorm": batchnorm_history.history,
    "Augmentation": augmentation_history.history,
    "Early Stopping": early_stop_history.history,
    "L2 Regularization": l2_history.history,
}

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, (name, h) in enumerate(histories.items()):
    ax = axes[idx]
    epochs_range = range(1, len(h["accuracy"]) + 1)
    ax.plot(epochs_range, h["accuracy"], "b-", label="Train Acc", linewidth=1.5)
    ax.plot(epochs_range, h["val_accuracy"], "r-", label="Val Acc", linewidth=1.5)
    gap = h["accuracy"][-1] - h["val_accuracy"][-1]
    ax.set_title(f"{name}\n(gap: {gap:.3f})", fontsize=12)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0.7, 1.0])

plt.suptitle("Individual Regularization Techniques - Accuracy Curves", fontsize=15)
plt.tight_layout()
plt.show()

## 10. Combined Model: All Techniques Together

Combine Dropout + BatchNormalization + Data Augmentation + L2 Regularization, trained with Early Stopping.

In [ ]:
def build_combined_model():
    """Full regularized model combining all techniques."""
    l2_reg = keras.regularizers.l2(1e-4)
    model = keras.Sequential([
        keras.layers.Input(shape=(28, 28, 1)),
        # Data augmentation
        keras.layers.RandomFlip("horizontal"),
        keras.layers.RandomRotation(0.1),
        # CNN feature extraction with BatchNorm and Dropout
        keras.layers.Conv2D(32, (3, 3), padding="same", kernel_regularizer=l2_reg),
        keras.layers.BatchNormalization(),
        keras.layers.Activation("relu"),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Dropout(0.25),
        keras.layers.Conv2D(64, (3, 3), padding="same", kernel_regularizer=l2_reg),
        keras.layers.BatchNormalization(),
        keras.layers.Activation("relu"),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Dropout(0.25),
        # Dense layers
        keras.layers.Flatten(),
        keras.layers.Dense(256, kernel_regularizer=l2_reg),
        keras.layers.BatchNormalization(),
        keras.layers.Activation("relu"),
        keras.layers.Dropout(0.5),
        keras.layers.Dense(128, kernel_regularizer=l2_reg),
        keras.layers.BatchNormalization(),
        keras.layers.Activation("relu"),
        keras.layers.Dropout(0.4),
        keras.layers.Dense(10, activation="softmax"),
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

print("Training combined regularized model...")
combined_model = build_combined_model()
combined_model.summary()

early_stopping_cb = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True, verbose=1
)

combined_history = combined_model.fit(
    x_train_img, y_train,
    epochs=50,
    batch_size=BATCH_SIZE,
    validation_data=(x_val_img, y_val),
    callbacks=[early_stopping_cb],
    verbose=1,
)

In [ ]:
# Compare baseline vs combined
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Baseline
bh = baseline_history.history
ep_b = range(1, len(bh["loss"]) + 1)
ax1.plot(ep_b, bh["accuracy"], "b--", label="Baseline Train", linewidth=1.5)
ax1.plot(ep_b, bh["val_accuracy"], "r--", label="Baseline Val", linewidth=1.5)

# Combined
ch = combined_history.history
ep_c = range(1, len(ch["loss"]) + 1)
ax1.plot(ep_c, ch["accuracy"], "b-", label="Combined Train", linewidth=2)
ax1.plot(ep_c, ch["val_accuracy"], "r-", label="Combined Val", linewidth=2)
ax1.set_title("Accuracy: Baseline vs Combined", fontsize=13)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(ep_b, bh["loss"], "b--", label="Baseline Train", linewidth=1.5)
ax2.plot(ep_b, bh["val_loss"], "r--", label="Baseline Val", linewidth=1.5)
ax2.plot(ep_c, ch["loss"], "b-", label="Combined Train", linewidth=2)
ax2.plot(ep_c, ch["val_loss"], "r-", label="Combined Val", linewidth=2)
ax2.set_title("Loss: Baseline vs Combined", fontsize=13)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nBaseline overfitting gap: {bh['accuracy'][-1] - bh['val_accuracy'][-1]:.4f}")
print(f"Combined overfitting gap: {ch['accuracy'][-1] - ch['val_accuracy'][-1]:.4f}")